# Notebook 03 — Topic Mining (LDA)

**Project:** CX Intelligence — NPS & Sentiment Analysis  
**Author:** Nicolás Zuleta Sierra

## Objectives
1. Discover recurring themes in banking complaints via LDA
2. Select optimal number of topics using coherence score
3. Name each topic with banking domain expertise
4. Identify the top 5 drivers of customer dissatisfaction
5. Cross-analyse topics with NPS and FinBERT sentiment

**Input:** `data/processed/features_nlp.csv`  
**Output:** `dominant_topic` column added · `models/lda_model.pkl` serialized

## 0. Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import spacy
from wordcloud import WordCloud

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.topics import (
    assign_topics,
    build_tfidf_matrix,
    calculate_coherence,
    get_topic_keywords,
    preprocess_for_lda,
    save_lda_model,
    train_lda,
)
from src.nps_calculator import nps_by_group

plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_colwidth", 100)

FEATURES_PATH = ROOT / "data" / "processed" / "features_nlp.csv"
LDA_MODEL_PATH = ROOT / "models" / "lda_model.pkl"

print(f"Features exist: {FEATURES_PATH.exists()}")

## 1. Load Features Dataset

In [ ]:
df = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(2)

## 2. Additional Preprocessing for LDA

We apply spaCy lemmatization with banking-domain stopwords on top of the
standard English set. Financial terms like 'bank', 'account', 'payment' are
so ubiquitous they would dominate all topics — removing them forces LDA to
discover the more discriminative themes.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

texts = df["lemmatized_text"].fillna("").tolist()

print("Running spaCy preprocessing for LDA…")
lda_texts = preprocess_for_lda(texts, nlp)

# Filter out empty results
df["lda_text"] = lda_texts
df_lda = df[df["lda_text"].str.strip() != ""].copy()
print(f"Rows with valid LDA text: {len(df_lda):,}")

# Example
print("\n--- LDA-preprocessed example ---")
print(df_lda["lda_text"].iloc[0][:200])

## 3. TF-IDF Vectorization

In [ ]:
tfidf_matrix, vectorizer = build_tfidf_matrix(df_lda["lda_text"].tolist())
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out()):,}")

## 4. Topic Count Selection — Coherence Score

We test k from 3 to 10. The optimal k maximizes the C_V coherence score,
which correlates well with human judgements of topic interpretability.

In [ ]:
print("Computing coherence scores for k=3 to 10 (this may take a few minutes)…")
coherence_results = calculate_coherence(df_lda["lda_text"].tolist(), range_topics=range(3, 11))

coherence_df = pd.DataFrame(coherence_results)
print(coherence_df)

fig = px.line(
    coherence_df, x="n_topics", y="coherence",
    markers=True,
    title="Coherence Score by Number of Topics",
    labels={"n_topics": "Number of Topics (k)", "coherence": "C_V Coherence"},
    color_discrete_sequence=["#1e40af"],
)
fig.show()

In [ ]:
# Select optimal k
optimal_k = coherence_df.loc[coherence_df["coherence"].idxmax(), "n_topics"]
print(f"Optimal number of topics: {optimal_k}")
print(f"Best coherence score: {coherence_df['coherence'].max():.4f}")

## 5. Train LDA Model

In [ ]:
N_TOPICS = int(optimal_k)  # Override here if you want a specific k

print(f"Training LDA with {N_TOPICS} topics…")
lda_model = train_lda(tfidf_matrix, n_topics=N_TOPICS)
print(f"Training complete. Log-likelihood: {lda_model.score(tfidf_matrix):.2f}")

## 6. Topic Keywords

In [ ]:
keywords = get_topic_keywords(lda_model, vectorizer, n_words=10)

print("=== Topic Keywords ===")
for topic_idx, words in keywords.items():
    print(f"Topic {topic_idx}: {', '.join(words)}")

## 7. Manual Topic Naming (Banking Domain Expertise)

Review the keywords above and assign descriptive names.  
Update the dictionary below based on what you see.

In [ ]:
# ⚠️  Update this mapping after reviewing topic keywords above
TOPIC_NAMES = {
    0: "Incorrect Charges & Unauthorized Debits",
    1: "Mortgage & Loan Servicing",
    2: "Customer Service & Communication",
    3: "Credit Reporting & Disputes",
    4: "Account Management & Closure",
    5: "Collection & Debt Recovery",
    6: "Foreclosure & Loss Mitigation",
    7: "Fraud & Identity Theft",
}

# Truncate to actual number of topics
TOPIC_NAMES = {k: v for k, v in TOPIC_NAMES.items() if k < N_TOPICS}

print("Topic name mapping:")
for k, v in TOPIC_NAMES.items():
    kws = ", ".join(keywords[k][:5])
    print(f"  [{k}] {v}\n       Keywords: {kws}")

## 8. Assign Topics to Complaints

In [ ]:
df_lda = assign_topics(df_lda, lda_model, vectorizer, text_col="lda_text")
df_lda["topic_name"] = df_lda["dominant_topic"].map(TOPIC_NAMES)

print("Topic distribution:")
print(df_lda["topic_name"].value_counts())

## 9. NPS by Topic

In [ ]:
nps_by_topic = nps_by_group(df_lda, "topic_name")
print(nps_by_topic[["topic_name", "n_total", "nps"]].sort_values("nps"))

fig = px.bar(
    nps_by_topic.sort_values("nps"),
    x="nps", y="topic_name", orientation="h",
    color="nps",
    color_continuous_scale=["#ef4444", "#f59e0b", "#22c55e"],
    title="NPS by Complaint Topic",
    labels={"nps": "NPS", "topic_name": ""},
    text="nps",
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(coloraxis_showscale=False, height=500)
fig.show()

## 10. FinBERT Sentiment by Topic

In [ ]:
sentiment_by_topic = (
    df_lda.groupby("topic_name")["finbert_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .reset_index()
)
sentiment_by_topic.columns = ["topic_name", "sentiment", "pct"]

fig = px.bar(
    sentiment_by_topic, x="topic_name", y="pct", color="sentiment",
    color_discrete_map={"positive": "#22c55e", "neutral": "#f59e0b", "negative": "#ef4444"},
    barmode="stack",
    title="FinBERT Sentiment Distribution by Topic",
    labels={"pct": "% of Complaints", "topic_name": "Topic"},
)
fig.update_layout(xaxis_tickangle=-30, height=450)
fig.show()

## 11. WordClouds by NPS Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
segment_pairs = [
    ("Promoter", "Blues", axes[0]),
    ("Detractor", "Reds", axes[1]),
]

for segment, colormap, ax in segment_pairs:
    subset_text = " ".join(df_lda[df_lda["nps_segment"] == segment]["lda_text"].dropna())
    if subset_text.strip():
        wc = WordCloud(
            width=800, height=500,
            background_color="white",
            colormap=colormap,
            max_words=100,
        ).generate(subset_text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"WordCloud — {segment}s", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## 12. Top 5 Drivers of Dissatisfaction

In [ ]:
top5_drivers = nps_by_topic.nsmallest(5, "nps")
print("=== TOP 5 DRIVERS OF DISSATISFACTION ===")
for i, (_, row) in enumerate(top5_drivers.iterrows(), 1):
    print(f"{i}. {row['topic_name']}")
    print(f"   NPS: {row['nps']} | Complaints: {row['n_total']:,} | "
          f"Detractors: {row['pct_detractors']:.1f}%")

## 13. Save Updated Features + LDA Model

In [ ]:
# Add topic columns back to the main features dataframe
# Re-merge on complaint_id
df_topics = df_lda[["complaint_id", "dominant_topic", "topic_name", "topic_probability", "lda_text"]].copy()

df_full = pd.read_csv(FEATURES_PATH, low_memory=False)
df_full = df_full.merge(df_topics, on="complaint_id", how="left")

df_full.to_csv(FEATURES_PATH, index=False)
print(f"Updated features saved: {len(df_full):,} rows")

# Save LDA model
save_lda_model(lda_model, vectorizer, LDA_MODEL_PATH)
print(f"LDA model + vectorizer saved to {LDA_MODEL_PATH}")

## 14. Conclusions

*(Fill in with actual values after running)*

**Top 5 Drivers of Customer Dissatisfaction:**
1. **[Topic 1]** — Lowest NPS, highest negative sentiment concentration
2. **[Topic 2]** — High complaint volume, slow resolution pattern
3. **[Topic 3]** — Friction beyond 30 days destroys NPS
4. **[Topic 4]** — Paper-heavy processes generate high dissatisfaction
5. **[Topic 5]** — High dissatisfaction at the account exit experience

**Key finding:** Topics with the lowest NPS are not necessarily the highest-volume topics.
Focus should be on topics with **both** high volume **and** low NPS for maximum CX impact.